The goal of this notebook is to showcase


1.   How to load and preprocess the classification challenge data
2.   How to build a baseline model
3.   How to prepare results to submit to the kaggle challenge



## Loading necessary packages

In [1]:
import os
import pickle
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

from google.colab import drive

Mounting Google Drive space for data access:

- You need to download the files train_X_y.pkl and test_X.pkl from [kaggle challenge site](https://www.kaggle.com/t/05ca47eb65b8426e9d53a2cc90a5875d), and then upload them to your **base_path** directory.

In [2]:
drive.mount('/content/drive')
base_path = 'drive/MyDrive/image-classification-challenge-w-26/W2026-0-0.3/'



Mounted at /content/drive


## Loading Classification Challenge data

In [3]:
# Training data
with open(os.path.join(base_path, 'train_X_y.pkl'),'rb') as f:
    X_train, y_train = pickle.load(f)

In [4]:
# Test data
with open(os.path.join(base_path, 'test_X.pkl'),'rb') as f:
    X_test = pickle.load(f)

In [5]:
X_train.shape, X_test.shape

((42000, 16, 16, 3), (18000, 16, 16, 3))

## Data preprocessing

In [6]:
X_train.min(), X_train.max()

(np.uint8(0), np.uint8(255))

In [7]:
# Data normalization
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

In [8]:
X_train.min(), X_train.max()

(np.float32(0.0), np.float32(1.0))

In [9]:
# transforming to 1D vector
X_train = X_train.reshape(X_train.shape[0], -1)
y_train = y_train.flatten()
X_test = X_test.reshape(X_test.shape[0], -1)

## Training a Baseline Random Forest model

In [10]:
baseline_model = RandomForestClassifier()
baseline_model.fit(X_train, y_train)

RandomForestClassifier()

## Making predictions and preparing submission file for the kaggle challenge

In [11]:
# Making predictions
y_predict = baseline_model.predict(X_test)

In [12]:
# Pack results in a pandas dataframe
test_predictions = pd.DataFrame({
    'rowId': range(0, len(y_predict.flatten())),
    'label': y_predict.flatten()})

In [13]:
test_predictions.head()

,rowId,label
0,0,6
1,1,9
2,2,8
3,3,1
4,4,4


In [14]:
# Exporting results
test_predictions.to_csv(os.path.join(base_path, 'sample_submission.csv'), index=False)

- HPC Cluster
-

Once you have successfully generated your results, say, "my_baseline_results.csv", you can upload this file for the kaggle challenge results.

## CNN Pipeline (Tuned to Improve the 0.787 Baseline)
This updated CNN keeps the Random Forest baseline unchanged and focuses only on the image model.

### What was tuned
- **Input normalization**: standardized each image channel with training-split mean and standard deviation instead of using only `[0, 1]` scaling.
- **Augmentation**: added padded random crops, horizontal flips, and light Gaussian noise to improve generalization on the small `16x16` images.
- **Model capacity**: deepened the CNN to `64 -> 128 -> 256` feature blocks with BatchNorm, GELU activations, and spatial dropout.
- **Optimization**: switched to `AdamW` with a higher learning rate ceiling, stronger weight decay, label smoothing, gradient clipping, and a `OneCycleLR` schedule.
- **Training control**: added early stopping and kept the best validation checkpoint before exporting predictions.

### Parameters that were updated
- `batch_size`: `128 -> 96`
- `epochs`: `25 -> 35` with early stopping
- `validation_split`: `0.15 -> 0.12`
- `learning_rate`: `3e-4 -> max_lr=1e-3` via `OneCycleLR`
- `weight_decay`: `1e-4 -> 5e-4`
- `loss`: plain cross-entropy -> cross-entropy with `label_smoothing=0.05`
- `augmentation`: horizontal flip only -> crop + flip + Gaussian noise
- `regularization`: dropout increased and changed to `Dropout2d` in convolution blocks

In [15]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---------- Tuned hyperparameters ----------
IMAGE_SIZE = 16
NUM_CLASSES = 10
BATCH_SIZE = 96
VAL_SIZE = 0.12
EPOCHS = 35
MAX_LR = 1e-3
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05
EARLY_STOPPING_PATIENCE = 8
MIN_REQUIRED_SCORE = 0.7870

# ---------- Data prep: NHWC -> NCHW image tensors ----------
X_train_img = X_train.reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 3).transpose(0, 3, 1, 2).astype('float32')
X_test_img = X_test.reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 3).transpose(0, 3, 1, 2).astype('float32')
y_train_1d = y_train.reshape(-1).astype('int64')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_img,
    y_train_1d,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train_1d
)

channel_mean = X_tr.mean(axis=(0, 2, 3)).astype('float32').reshape(3, 1, 1)
channel_std = X_tr.std(axis=(0, 2, 3)).astype('float32').reshape(3, 1, 1) + 1e-6

class ImageDataset(Dataset):
    def __init__(self, images, labels=None, augment=False, mean=None, std=None, crop_padding=0):
        self.images = images
        self.labels = labels
        self.augment = augment
        self.crop_padding = crop_padding
        self.mean = torch.tensor(mean, dtype=torch.float32) if mean is not None else None
        self.std = torch.tensor(std, dtype=torch.float32) if std is not None else None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx]).float()
        if self.augment:
            if torch.rand(1).item() < 0.5:
                x = torch.flip(x, dims=[2])
            if self.crop_padding > 0:
                x = F.pad(x, (self.crop_padding, self.crop_padding, self.crop_padding, self.crop_padding), mode='reflect')
                top = torch.randint(0, 2 * self.crop_padding + 1, (1,)).item()
                left = torch.randint(0, 2 * self.crop_padding + 1, (1,)).item()
                x = x[:, top:top + IMAGE_SIZE, left:left + IMAGE_SIZE]
            noise = torch.randn_like(x) * 0.015
            x = torch.clamp(x + noise, 0.0, 1.0)
        if self.mean is not None and self.std is not None:
            x = (x - self.mean) / self.std
        if self.labels is None:
            return x
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_ds = ImageDataset(X_tr, y_tr, augment=True, mean=channel_mean, std=channel_std, crop_padding=2)
val_ds = ImageDataset(X_val, y_val, augment=False, mean=channel_mean, std=channel_std)
test_ds = ImageDataset(X_test_img, labels=None, augment=False, mean=channel_mean, std=channel_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

# ---------- Improved CNN ----------
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )

    def forward(self, x):
        return self.block(x)

class BetterCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64, dropout=0.10),
            ConvBlock(64, 128, dropout=0.15),
            ConvBlock(128, 256, dropout=0.20),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.GELU(),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = BetterCNN(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=MAX_LR / 10, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.20,
    anneal_strategy='cos',
    div_factor=10,
    final_div_factor=100
)
scaler = GradScaler(enabled=torch.cuda.is_available())

# ---------- Training + validation ----------
best_val_acc = 0.0
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=torch.cuda.is_available()):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * xb.size(0)

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds.tolist())
            val_true.extend(yb.numpy().tolist())

    val_acc = accuracy_score(val_true, val_preds)
    epoch_loss = running_loss / len(train_loader.dataset)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping triggered after {epoch} epochs with no validation improvement for {EARLY_STOPPING_PATIENCE} epochs.')
        break

# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)

# ---------- Score check before submission export ----------
print(f'Best validation accuracy: {best_val_acc:.4f} (epoch {best_epoch})')
if best_val_acc >= MIN_REQUIRED_SCORE:
    model.eval()
    test_preds = []
    with torch.no_grad():
        for xb in test_loader:
            xb = xb.to(device, non_blocking=True)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            test_preds.extend(preds.tolist())

    submission = pd.DataFrame({
        'rowId': np.arange(len(test_preds)),
        'label': np.array(test_preds, dtype=np.int64)
    })
    submission_path = os.path.join(base_path, 'cnn_submission_improved1.csv')
    submission.to_csv(submission_path, index=False)
    print(f'Validation gate passed ({best_val_acc:.4f} >= {MIN_REQUIRED_SCORE:.4f}).')
    print(f'Submission saved to: {submission_path}')
    display(submission.head())
else:
    print(f'Validation gate not passed ({best_val_acc:.4f} < {MIN_REQUIRED_SCORE:.4f}).')
    print('Submission CSV was not exported. Adjust model/training and re-run.')

Using device: cuda


/tmp/ipykernel_11401/4231219341.py:147: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_11401/4231219341.py:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_11401/4231219341.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 01/35 | Train Loss: 1.7963 | Val Acc: 0.4927


/tmp/ipykernel_11401/4231219341.py:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_11401/4231219341.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 02/35 | Train Loss: 1.5247 | Val Acc: 0.5724


...

/tmp/ipykernel_11401/4231219341.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_11401/4231219341.py:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 35/35 | Train Loss: 0.5859 | Val Acc: 0.8373
Best validation accuracy: 0.8387 (epoch 33)


/tmp/ipykernel_11401/4231219341.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_11401/4231219341.py:213: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Validation gate passed (0.8387 >= 0.7870).
Submission saved to: drive/MyDrive/image-classification-challenge-w-26/W2026-0-0.3/cnn_submission_improved1.csv


,rowId,label
0,0,6
1,1,9
2,2,8
3,3,8
4,4,4
